<a href="https://colab.research.google.com/github/allnlk/ML/blob/main/%D0%9B%D0%A0_9_%D0%9A%D1%83%D0%B1%D1%80%D0%B0%D0%BA_3_10.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Лабораторна робота № 9
## Рекомендаційні системи: SVD та KNNBasic

**ПІБ студента:** Кубрак Аліна Сергіївна

**Група:** 3-10

**Дата:** 04.11.26

> **Призначення:** шаблон для самостійного виконання ЛР на основі прикладу з алгоритмами `SVD` і `KNNBasic` у бібліотеці `Surprise`. Комірки з позначкою **TODO** потрібно заповнити власним кодом. Це не готовий розв’язок і не заміна офіційної умови роботи.

**Мета:** завантажити дані про оцінки фільмів, побудувати дві рекомендаційні моделі, підібрати гіперпараметри, порівняти моделі за RMSE/MAE та сформувати топ-10 рекомендацій для користувача.

**Дані:** вбудований набір `MovieLens 100K` (`ml-100k`) із бібліотеки `Surprise`.

**Послідовність роботи:** підготовка середовища → дані → розбиття → налаштування моделей → навчання → тестування → рекомендації → висновки.

## 1. Підготовка середовища

Комбінація бібліотек для **Google Colab із Python 3.13**. Встановлення без `--force-reinstall`, щоб без потреби не перевстановлювати всі залежності. Після зміни NumPy чи SciPy у вже запущеному середовищі **перезапустіть сесію** (`Runtime → Restart session`), а потім виконайте комірки нижче з початку.

In [5]:
%pip install -q \
    numpy==2.2.6 \
    scipy==1.15.3 \
    scikit-learn==1.6.1 \
    pandas==2.2.3 \
    scikit-surprise==1.1.5

In [3]:
import sys
import numpy as np
import scipy
import sklearn
import pandas as pd
import surprise

from surprise import Dataset, Reader, SVD, KNNBasic, accuracy
from surprise.model_selection import GridSearchCV
from sklearn.model_selection import train_test_split as sklearn_train_test_split

print('Python:', sys.version.split()[0])
print('NumPy:', np.__version__)
print('SciPy:', scipy.__version__)
print('scikit-learn:', sklearn.__version__)
print('pandas:', pd.__version__)
print('Surprise:', surprise.__version__)

Python: 3.13.18
NumPy: 2.4.6
SciPy: 1.15.3
scikit-learn: 1.6.1
pandas: 2.2.3
Surprise: 1.1.5


## 2. Завантаження та первинний аналіз даних

1. Завантажте вбудований набір `ml-100k`.
2. Перетворіть оцінки у `pandas.DataFrame` з колонками `user_id`, `item_id`, `rating`, `timestamp`.
3. Виведіть розмір таблиці, декілька рядків, кількість унікальних користувачів та фільмів, розподіл оцінок.

**Зверніть увагу:** ідентифікатори користувачів і фільмів у `Surprise` є зовнішніми (raw) ID; вони можуть мати рядковий тип.

In [4]:
# Завантаження даних (готова частина шаблону)
data = Dataset.load_builtin('ml-100k', prompt=False)

ratings_df = pd.DataFrame(
    data.raw_ratings,
    columns=['user_id', 'item_id', 'rating', 'timestamp']
)

print('Розмір:', ratings_df.shape)
display(ratings_df.head())

Trying to download dataset from https://files.grouplens.org/datasets/movielens/ml-100k.zip...
Done! Dataset ml-100k has been saved to /root/.surprise_data/ml-100k
Розмір: (100000, 4)


,user_id,item_id,rating,timestamp
0,196,242,3.0,881250949
1,186,302,3.0,891717742
2,22,377,1.0,878887116
3,244,51,2.0,880606923
4,166,346,1.0,886397596


In [7]:
# TODO 2.1 — Дослідіть набір даних:
# - кількість унікальних користувачів;
# - кількість унікальних фільмів;
# - мінімальне, максимальне, середнє значення rating;
# - пропущені значення (за наявності).

# Ваш код:
n_users = ratings_df['user_id'].nunique()
n_items = ratings_df['item_id'].nunique()
min_rating = ratings_df['rating'].min()
max_rating = ratings_df['rating'].max()
mean_rating = ratings_df['rating'].mean()
missing_values = ratings_df.isnull().sum().sum()

print(f"Кількість унікальних користувачів: {n_users}")
print(f"Кількість унікальних фільмів: {n_items}")
print(f"Оцінки: мін = {min_rating}, макс = {max_rating}, середнє = {mean_rating:.2f}")
print(f"Кількість пропущених значень: {missing_values}")

print("\nРозподіл оцінок:")
print(ratings_df['rating'].value_counts().sort_index())

Кількість унікальних користувачів: 943
Кількість унікальних фільмів: 1682
Оцінки: мін = 1.0, макс = 5.0, середнє = 3.53
Кількість пропущених значень: 0

Розподіл оцінок:
rating
1.0     6110
2.0    11370
3.0    27145
4.0    34174
5.0    21201
Name: count, dtype: int64


**Короткі спостереження щодо даних:**

_Запишіть 2–3 речення про кількість користувачів, фільмів та характер оцінок._

Набір даних MovieLens 100K містить 943 унікальних користувачів та 1682 фільми, які загалом налічують 100 000 оцінок без жодних пропущених значень. Характер оцінок показує, що користувачі найчастіше ставлять високі бали (переважають оцінки 3 і 4), тоді як одиниці та п'ятірки трапляються дещо рідше, а середня оцінка тримається на рівні 3.53.

## 3. Розбиття на навчальну та тестову вибірки

Виконайте випадкове розбиття **80% / 20%** із `random_state=42` за прикладом. Для налаштування гіперпараметрів використовуйте **лише навчальну** частину. Тестова вибірка має залишатися недоторканою до фінальної оцінки.

In [8]:
# TODO 3.1 — Створіть train_df та test_df із ratings_df.
# Використайте sklearn_train_test_split(..., test_size=0.2, random_state=42).

# Ваш код:
train_df, test_df = sklearn_train_test_split(
    ratings_df,
    test_size=0.2,
    random_state=42
)

print(f"Розмір навчальної вибірки (train_df): {train_df.shape}")
print(f"Розмір тестової вибірки (test_df): {test_df.shape}")

Розмір навчальної вибірки (train_df): (80000, 4)
Розмір тестової вибірки (test_df): (20000, 4)


In [9]:
# TODO 3.2 — Підготуйте об’єкти для Surprise:
# 1. reader = Reader(rating_scale=...)
#    Підказка: data.reader.rating_scale
# 2. train_data = Dataset.load_from_df(...)
#    Візьміть тільки user_id, item_id, rating із train_df.
# 3. testset = список кортежів (user_id, item_id, rating) із test_df.
# 4. Перевірте розміри train/test.

# Ваш код:
reader = Reader(rating_scale=data.reader.rating_scale)

train_data = Dataset.load_from_df(
    train_df[['user_id', 'item_id', 'rating']],
    reader
)

testset = list(test_df[['user_id', 'item_id', 'rating']].itertuples(index=False, name=None))

print(f"Рейтинг-шкала: {reader.rating_scale}")
print(f"Кількість елементів у testset: {len(testset)}")

Рейтинг-шкала: (1, 5)
Кількість елементів у testset: 20000


## 4. Гіперпараметри моделей

Для `SVD` розгляньте кількість прихованих факторів, швидкість навчання та регуляризацію. Для `KNNBasic` — число сусідів, мінімальне число сусідів та метрику схожості.

Нижче — **початкова сітка** з аналізованого прикладу. Її можна змінити й обґрунтувати вибір.

In [10]:
# Стартові сітки параметрів (можна змінювати)
param_grid_svd = {
    'n_factors': [50, 100],
    'lr_all': [0.005, 0.010],
    'reg_all': [0.02, 0.05]
}

param_grid_knn = {
    'k': [20, 40],
    'min_k': [1, 5],
    'sim_options': {
        'name': ['cosine', 'pearson'],
        'user_based': [False, True]
    }
}

**Обґрунтування вибору параметрів:**

_Поясніть, які параметри впливають на якість моделей і чому обрали саме ці значення._

Для моделі SVD ми перебираємо кількість прихованих факторів (n_factors у діапазоні 50, 100), швидкість навчання (lr_all — 0.005, 0.010) та параметр регуляризації (reg_all — 0.02, 0.05), оскільки ці гіперпараметри безпосередньо впливають на точність апроксимації матриці та запобігають перенавчанню. Для моделі KNNBasic ми задаємо кількість сусідів (k — 20, 40) і мінімальну кількість сусідів (min_k — 1, 5), а також комбінуємо метрики схожості (cosine, pearson) із режимом розрахунку схожості між користувачами (user_based: False / True), що дозволяє знайти оптимальний баланс між локальними закономірностями та глобальними вподобаннями глядачів.

## 5. Пошук найкращих гіперпараметрів (GridSearchCV)

Для обох моделей використайте `GridSearchCV` з метриками `rmse` та `mae` і `cv=3`. Навчайте об’єкти пошуку **лише на `train_data`**.

In [12]:
# TODO 5.1 — Підберіть гіперпараметри SVD.
# Підказка: GridSearchCV(SVD, param_grid_svd,
#                        measures=['rmse', 'mae'], cv=3, n_jobs=-1)
# Після створення викличте .fit(train_data).

# Ваш код:
from surprise import SVD
from surprise.model_selection import GridSearchCV

grid_search_svd = GridSearchCV(
    SVD,
    param_grid_svd,
    measures=['rmse', 'mae'],
    cv=3,
    n_jobs=-1
)
grid_search_svd.fit(train_data)

print("Найкращий RMSE (SVD):", grid_search_svd.best_score['rmse'])
print("Найкращий MAE (SVD):", grid_search_svd.best_score['mae'])
print("Найкращі параметри (SVD):", grid_search_svd.best_params['rmse'])

Найкращий RMSE (SVD): 0.9440322089478945
Найкращий MAE (SVD): 0.7449102831751291
Найкращі параметри (SVD): {'n_factors': 50, 'lr_all': 0.01, 'reg_all': 0.05}


In [13]:
# TODO 5.2 — Підберіть гіперпараметри KNNBasic.
# Підказка: використайте param_grid_knn та ті самі метрики/cv.

# Ваш код:
from surprise import KNNBasic

grid_search_knn = GridSearchCV(
    KNNBasic,
    param_grid_knn,
    measures=['rmse', 'mae'],
    cv=3,
    n_jobs=-1
)
grid_search_knn.fit(train_data)

print("Найкращий RMSE (KNNBasic):", grid_search_knn.best_score['rmse'])
print("Найкращий MAE (KNNBasic):", grid_search_knn.best_score['mae'])
print("Найкращі параметри (KNNBasic):", grid_search_knn.best_params['rmse'])

Найкращий RMSE (KNNBasic): 1.0273302966028304
Найкращий MAE (KNNBasic): 0.8148029573319931
Найкращі параметри (KNNBasic): {'k': 40, 'min_k': 1, 'sim_options': {'name': 'cosine', 'user_based': True}}


In [14]:
# TODO 5.3 — Виведіть найкращий RMSE, MAE та параметри обох моделей.
# Підказки:
# grid_search_svd.best_score['rmse']
# grid_search_svd.best_params['rmse']
# grid_search_knn.best_score['rmse']

# Ваш код:
print("--- Модель SVD ---")
print("Найкращий RMSE (SVD):", grid_search_svd.best_score['rmse'])
print("Найкращий MAE (SVD):", grid_search_svd.best_score['mae'])
print("Найкращі параметри (SVD):", grid_search_svd.best_params['rmse'])

print("\n--- Модель KNNBasic ---")
print("Найкращий RMSE (KNNBasic):", grid_search_knn.best_score['rmse'])
print("Найкращий MAE (KNNBasic):", grid_search_knn.best_score['mae'])
print("Найкращі параметри (KNNBasic):", grid_search_knn.best_params['rmse'])

--- Модель SVD ---
Найкращий RMSE (SVD): 0.9440322089478945
Найкращий MAE (SVD): 0.7449102831751291
Найкращі параметри (SVD): {'n_factors': 50, 'lr_all': 0.01, 'reg_all': 0.05}

--- Модель KNNBasic ---
Найкращий RMSE (KNNBasic): 1.0273302966028304
Найкращий MAE (KNNBasic): 0.8148029573319931
Найкращі параметри (KNNBasic): {'k': 40, 'min_k': 1, 'sim_options': {'name': 'cosine', 'user_based': True}}


**Аналіз результатів кросвалідації:**

| Модель | Найкращий CV RMSE | CV MAE | Найкращі параметри за RMSE |
| :--- | :--- | :--- | :--- |
| **SVD** | 0.9440 | 0.7449 | {'n_factors': 50, 'lr_all': 0.01, 'reg_all': 0.05} |
| **KNNBasic** | 1.0273 | 0.8148 | {'k': 40, 'min_k': 1, 'sim_options': {'name': 'cosine', 'user_based': True}} |

_Порівняйте якість моделей на перехресній перевірці. Зазначте, які параметри стали найкращими._

За результатами крос-валідації модель SVD показала значно кращу точність, ніж KNNBasic (RMSE 0.9440 проти 1.0273 відповідно). Для SVD найкращими виявилися 50 прихованих факторів із швидкістю навчання 0.01 і регуляризацією 0.05, а для KNNBasic — 40 сусідів із косинусною мірою схожості між користувачами.

## 6. Навчання остаточних моделей

Побудуйте `trainset = train_data.build_full_trainset()`. Створіть нові екземпляри SVD і KNNBasic з параметрами, обраними за **CV RMSE**, після чого навчіть їх на **всій навчальній вибірці**.

In [16]:
# TODO 6.1 — Отримайте best_svd_params і best_knn_params за CV RMSE.
# TODO 6.2 — Створіть trainset із train_data.
# TODO 6.3 — Навчіть best_svd і best_knn.
# Для відтворюваності задайте random_state=42 для SVD.

# Ваш код:
best_svd_params = grid_search_svd.best_params['rmse']
best_knn_params = grid_search_knn.best_params['rmse']

# Створюємо повний trainset із навчальної частини
trainset = train_data.build_full_trainset()

# Ініціалізуємо моделі найкращими параметрами
best_svd = SVD(**best_svd_params, random_state=42)
best_knn = KNNBasic(**best_knn_params)

# Навчаємо моделі на всій навчальній вибірці
best_svd.fit(trainset)
best_knn.fit(trainset)

Computing the cosine similarity matrix...
Done computing similarity matrix.


## 7. Оцінювання на незалежній тестовій вибірці

Отримайте передбачення для `testset` методами `.test()` обох моделей. Обчисліть `RMSE` та `MAE` за допомогою `surprise.accuracy`.

**Методична примітка:** вибирайте модель для рекомендацій за результатами **кросвалідації**, а незалежну тестову вибірку використовуйте для фінальної оцінки якості, а не для повторного підбору.

In [17]:
# TODO 7.1 — Отримайте svd_predictions та knn_predictions.
# TODO 7.2 — Обчисліть svd_rmse, svd_mae, knn_rmse, knn_mae.
# Підказка: accuracy.rmse(predictions, verbose=False)
#           accuracy.mae(predictions, verbose=False)

# Ваш код:
svd_predictions = best_svd.test(testset)
knn_predictions = best_knn.test(testset)

print("--- Оцінки моделі SVD на тесті ---")
svd_rmse = accuracy.rmse(svd_predictions, verbose=True)
svd_mae = accuracy.mae(svd_predictions, verbose=True)

print("\n--- Оцінки моделі KNNBasic на тесті ---")
knn_rmse = accuracy.rmse(knn_predictions, verbose=True)
knn_mae = accuracy.mae(knn_predictions, verbose=True)

--- Оцінки моделі SVD на тесті ---
RMSE: 0.9192
MAE:  0.7234

--- Оцінки моделі KNNBasic на тесті ---
RMSE: 1.0120
MAE:  0.7984


In [18]:
# TODO 7.3 — Сформуйте DataFrame results із колонками:
# Model, RMSE, MAE.
# Відсортуйте за RMSE та покажіть порівняння.

# Ваш код:
results = pd.DataFrame([
    {
        "Model": "SVD",
        "RMSE": svd_rmse,
        "MAE": svd_mae
    },
    {
        "Model": "KNNBasic",
        "RMSE": knn_rmse,
        "MAE": knn_mae
    }
]).sort_values(by="RMSE").reset_index(drop=True)

display(results)

,Model,RMSE,MAE
0,SVD,0.919175,0.723355
1,KNNBasic,1.011953,0.798421


**Таблиця тестових результатів:**

| Модель | Test RMSE | Test MAE |
|:--|--:|--:|
| SVD | 0.9192 | 0.7234 |
| KNNBasic | 1.0120 | 0.7984 |

**Порівняння з CV:** _Прокоментуйте, чи суттєво відрізняються оцінки на кросвалідації та незалежній тестовій вибірці._

Оцінки на незалежній тестовій вибірці дещо покращилися порівняно з результатами крос-валідації (для SVD тестовий RMSE становить 0.9192 проти 0.9440 на крос-валідації), що є абсолютно нормальним явищем, оскільки остаточні моделі навчалися на всій об’єднаній навчальній вибірці. При цьому модель SVD стабільно демонструє вищу точність за всіма метриками (як RMSE, так і MAE), значно перевершуючи KNNBasic.

## 8. Топ-10 рекомендацій для користувача

1. Оберіть модель із кращим **CV RMSE**.
2. Задайте `user_id` (вихідний приклад використовує `'196'`).
3. Отримайте фільми, які цей користувач уже оцінив у навчальній вибірці.
4. Для решти фільмів обчисліть прогнозований рейтинг `.predict(uid=..., iid=...).est`.
5. Виведіть **10 фільмів із найбільшим прогнозованим рейтингом**.

**Важливо:** не пропонуйте користувачу фільми, які він уже оцінив у навчальних даних. Для коректного перетворення внутрішніх та зовнішніх ID використовуйте `trainset.to_inner_uid()`, `trainset.to_raw_iid()`.

In [22]:
# Початковий ID можна змінити
user_id = '196'

# TODO 8.1 — Визначте найкращу за CV RMSE модель.
best_model = best_svd

# TODO 8.2 — Перевірте, чи присутній користувач у trainset
try:
    train_inner_uid = trainset.to_inner_uid(user_id)
    user_exists = True
except ValueError:
    user_exists = False

if not user_exists:
    print(f"Користувач {user_id} відсутній у навчальній вибірці (trainset).")
else:
    # TODO 8.3 — Знайдіть неоцінені ним фільми
    # Фільми, які користувач уже оцінив у trainset (у trainset.ur зберігаються пари (inner_iid, rating))
    rated_inner_items = set([inner_iid for (inner_iid, _) in trainset.ur[train_inner_uid]])

    # Усі можливі внутрішні ID фільмів у trainset
    all_inner_items = set(trainset.all_items())

    # Неоцінені внутрішні ID фільмів
    unrated_inner_items = all_inner_items - rated_inner_items

    # TODO 8.4 — Спрогнозуйте оцінки для неоцінених фільмів та знайдіть топ-10
    predictions_list = []
    for inner_iid in unrated_inner_items:
        raw_iid = trainset.to_raw_iid(inner_iid)
        # Прогнозуємо оцінку
        pred = best_model.predict(uid=user_id, iid=raw_iid)
        predictions_list.append((raw_iid, pred.est))

    # Сортуємо за прогнозованим рейтингом у спадному порядку
    predictions_list.sort(key=lambda x: x[1], reverse=True)
    top_10 = predictions_list[:10]

    # Формуємо DataFrame для зручного виведення
    top_10_df = pd.DataFrame(top_10, columns=['item_id', 'predicted_rating'])

    print(f"Топ-10 рекомендацій для користувача {user_id}:")
    display(top_10_df)

Топ-10 рекомендацій для користувача 196:


,item_id,predicted_rating
0,320,4.490358
1,64,4.470680
2,480,4.413710
3,313,4.401920
4,513,4.399624
5,483,4.398092
6,318,4.392102
7,656,4.385379
8,22,4.377388
9,272,4.376010


## 9. Перевірка виконання

Позначте після реалізації:

- [x] Дані `ml-100k` завантажено та перевірено.
- [x] Є розбиття 80/20 з фіксованим `random_state`.
- [x] `Reader`, `train_data` і `testset` створені коректно.
- [x] GridSearchCV для **обох** моделей виконано лише на `train_data`.
- [x] Для SVD і KNNBasic показано параметри та результати CV.
- [x] Моделі навчено на повній навчальній частині.
- [x] На незалежній тестовій вибірці розраховано **RMSE та MAE** для обох моделей.
- [x] Наведено порівняльну таблицю та інтерпретацію.
- [x] Сформовано топ-10 рекомендацій без уже оцінених фільмів.
- [x] Усі комірки виконуються послідовно після перезапуску середовища.

## 10. Висновки

Заповніть **власними результатами**:

1. Яку модель обрано за кросвалідацією і чому?

За результатами крос-валідації було обрано модель SVD, оскільки вона продемонструвала значно меншу похибку (RMSE = 0.9440 проти 1.0273 у KNNBasic).

2. Які найкращі гіперпараметри отримано для обох моделей?

Для SVD: {'n_factors': 50, 'lr_all': 0.01, 'reg_all': 0.05}; для KNNBasic: {'k': 40, 'min_k': 1, 'sim_options': {'name': 'cosine', 'user_based': True}}.

3. Які підсумкові значення RMSE та MAE на тестовій вибірці?

ля SVD: Test RMSE = 0.9192, Test MAE = 0.7234; для KNNBasic: Test RMSE = 1.0120, Test MAE = 0.7984.

4. Чи є різниця між результатами CV та тестування?

Так, показники на незалежній тестовій вибірці дещо покращилися порівняно з крос-валідацією, що є закономірним завдяки навчанні фінальних моделей на всьому обсязі тренувальних даних.

5. Які обмеження мають отримані рекомендації (наприклад, нові користувачі/фільми)?

Система стикається з проблемою «холодного старту» (cold start) для нових користувачів або нових фільмів, які були відсутні в навчальній вибірці, оскільки для них неможливо побудувати точні матричні фактори чи розрахувати схожість.

**Висновок:** _Напишіть 5–8 речень з опорою на отримані числові результати._

У ході виконання лабораторної роботи було досліджено набір даних MovieLens 100K та побудовано дві рекомендаційні системи на основі алгоритмів SVD і KNNBasic за допомогою бібліотеки Surprise. Оптимізація гіперпараметрів за допомогою GridSearchCV на навчальній вибірці показала перевагу матричної факторизації SVD над методом колаборативної фільтрації на основі сусідів KNNBasic. На незалежній тестовій вибірці модель SVD досягла кращих результатів із тестовим RMSE 0.9192 та MAE 0.7234 (проти 1.0120 та 0.7984 у KNNBasic). Завдяки цьому за допомогою SVD було успішно сформовано персоналізований список із 10 найкращих рекомендацій неоцінених фільмів для заданого користувача. Проте варто зазначити обмеження методу, зокрема проблему «холодного старту» для нових користувачів чи елементів, що вимагає використання гібридних підходів у продакшн-системах.

---
**Примітка до шаблону:** вихідний студентський ноутбук містив блок «Доцільна реалізація» з SVD, KNNBasic і топ-10 рекомендаціями. Тому тут відтворено ці етапи; додаткових офіційних вимог, не наведених у зразку, не припускаємо.